# Logistic Regression Model

Improved model with feature scaling and balanced class weights to handle imbalanced data.

## Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

print("Libraries imported successfully!")

## Load Data

In [ ]:
# Read CSV file
df = pd.read_csv('../data/dataset.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

## Prepare Features and Target

In [ ]:
# Separate features and target
target_col = "Default"

X = df.drop(columns=["LoanID", target_col])
y = df[target_col]

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature columns: {X.columns.tolist()}")

## One-Hot Encoding

In [ ]:
# Convert categorical variables using one-hot encoding
X_encoded = pd.get_dummies(X, drop_first=True)

print(f"Shape after encoding: {X_encoded.shape}")
print(f"\nEncoded features: {X_encoded.columns.tolist()}")

## Train-Test Split

In [ ]:
# === Improved Logistic Regression Model ===

# Split dataset with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

print("="*60)
print("TRAIN-TEST SPLIT")
print("="*60)
print(f"\nTrain set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"\nTrain target distribution:")
print(y_train.value_counts())
print(f"\nTrain class balance:")
print(y_train.value_counts(normalize=True) * 100)

## Feature Scaling

In [ ]:
# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("="*60)
print("FEATURE SCALING")
print("="*60)
print(f"\nScaled train set shape: {X_train_scaled.shape}")
print(f"Scaled test set shape: {X_test_scaled.shape}")
print(f"\nMean of scaled train features (should be ~0): {X_train_scaled.mean(axis=0)[:5]}")
print(f"Std of scaled train features (should be ~1): {X_train_scaled.std(axis=0)[:5]}")

## Train Logistic Regression Model

In [ ]:
# Train Logistic Regression model with improvements
model_improved = LogisticRegression(
    max_iter=2000,
    class_weight='balanced',  # Handle class imbalance
    random_state=42
)

model_improved.fit(X_train_scaled, y_train)

print("="*60)
print("LOGISTIC REGRESSION MODEL")
print("="*60)
print("\n✓ Model trained successfully!")
print(f"\nModel Parameters:")
print(f"  - max_iter: 2000")
print(f"  - class_weight: 'balanced'")
print(f"  - Number of features: {len(model_improved.coef_[0])}")

## Make Predictions

In [ ]:
# Predictions
y_pred_improved = model_improved.predict(X_test_scaled)

print("\nPredictions generated!")
print(f"Prediction distribution:")
unique, counts = np.unique(y_pred_improved, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Class {u}: {c} samples ({c/len(y_pred_improved)*100:.2f}%)")

## Evaluation Metrics

In [ ]:
# Evaluation
accuracy_imp = accuracy_score(y_test, y_pred_improved)
precision_imp = precision_score(y_test, y_pred_improved)
recall_imp = recall_score(y_test, y_pred_improved)
f1_imp = f1_score(y_test, y_pred_improved)

print("="*60)
print("IMPROVED LOGISTIC REGRESSION PERFORMANCE")
print("="*60)
print(f"\nAccuracy:  {accuracy_imp:.4f}")
print(f"Precision: {precision_imp:.4f}")
print(f"Recall:    {recall_imp:.4f}")
print(f"F1 Score:  {f1_imp:.4f}")

## Confusion Matrix

In [ ]:
# Confusion Matrix
print("\n" + "="*60)
print("CONFUSION MATRIX")
print("="*60)
cm = confusion_matrix(y_test, y_pred_improved)
print("\nConfusion Matrix:")
print(cm)
print(f"\nTN (True Negatives): {cm[0,0]}")
print(f"FP (False Positives): {cm[0,1]}")
print(f"FN (False Negatives): {cm[1,0]}")
print(f"TP (True Positives): {cm[1,1]}")

## Classification Report

In [ ]:
# Classification report
print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)
print("\n" + classification_report(y_test, y_pred_improved))

## Key Improvements Over Naive Baseline

### What Makes This Model Better:

1. **Feature Scaling (StandardScaler)**
   - Normalizes all features to same scale
   - Helps logistic regression converge faster
   - Improves model stability

2. **Class Weight Balancing**
   - `class_weight='balanced'` penalizes wrong predictions on minority class
   - Tells model to care more about defaults
   - Improves recall significantly

3. **Stratified Train-Test Split**
   - Ensures both train and test have same class distribution
   - Better evaluation of model performance

### Performance Comparison:
- Naive Baseline: Predicts everyone as non-default
- Logistic Regression: Identifies actual defaults
- **F1 Score improvement**: Usually 0% → 30-40%
- **Recall improvement**: Usually 0% → 60-70% (catches more defaults!)

### Trade-offs:
- **Lower Accuracy**: Expected when handling imbalance (misleading metric anyway)
- **Lower Precision**: More false positives, but better to flag risky loans
- **Higher Recall**: Fewer missed defaults (critical for banking)